# 🛒 Google Data Analytics Capstone — Case Study 3
## Step 4: ANALYZE — E-Commerce Customer Buying Behavior
**Analyst:** [Your Name]  
**Date:** April 2026  
**Dataset:** Online Retail II — UCI Machine Learning Repository  
**Tool:** Python (Google Colab)  

---
### Business Question
> *What patterns exist in customer purchasing behavior, and how can these insights help the business better serve its customers and grow revenue?*

---

## 📦 SECTION 1: Install & Import Libraries

In [ ]:
# Install any missing libraries
!pip install openpyxl --quiet

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Chart styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('✅ All libraries loaded successfully!')

## 📂 SECTION 2: Load Your Data
**Instructions:**
1. Upload your cleaned CSV from Google Sheets
2. Go to the Files panel (folder icon on left) → Upload
3. Upload your file named `online_retail_clean.csv`

In [ ]:
# ----- OPTION A: Load from uploaded CSV (recommended) -----
df = pd.read_csv('online_retail_clean.csv', encoding='latin1')

# ----- OPTION B: Load directly from Excel -----
# df1 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010', engine='openpyxl')
# df2 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011', engine='openpyxl')
# df = pd.concat([df1, df2], ignore_index=True)

print(f'✅ Data loaded: {df.shape[0]:,} rows and {df.shape[1]} columns')
df.head()

## 🔍 SECTION 3: Data Exploration

In [ ]:
# Dataset overview
print('=== DATASET OVERVIEW ===')
print(f'Rows:    {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print()
print('=== COLUMN DATA TYPES ===')
print(df.dtypes)
print()
print('=== MISSING VALUES ===')
print(df.isnull().sum())

In [ ]:
# Statistical summary
print('=== STATISTICAL SUMMARY ===')
df[['Quantity', 'UnitPrice']].describe().round(2)

## 🧹 SECTION 4: Data Cleaning in Python
*(Skip if you already cleaned in Google Sheets — just run the Revenue column and date conversion)*

In [ ]:
# Step 1: Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Step 2: Remove cancellations (InvoiceNo starts with 'C')
before = len(df)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Removed {before - len(df):,} cancelled orders')

# Step 3: Remove negative or zero quantities
before = len(df)
df = df[df['Quantity'] > 0]
print(f'Removed {before - len(df):,} rows with invalid Quantity')

# Step 4: Remove zero or negative prices
before = len(df)
df = df[df['UnitPrice'] > 0]
print(f'Removed {before - len(df):,} rows with invalid UnitPrice')

# Step 5: Add Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Step 6: Add time-based columns
df['Year']       = df['InvoiceDate'].dt.year
df['Month']      = df['InvoiceDate'].dt.month
df['MonthName']  = df['InvoiceDate'].dt.strftime('%b')
df['DayOfWeek']  = df['InvoiceDate'].dt.strftime('%A')
df['Hour']       = df['InvoiceDate'].dt.hour
df['YearMonth']  = df['InvoiceDate'].dt.to_period('M')

# Step 7: Customer type
df['CustomerType'] = df['CustomerID'].apply(lambda x: 'Guest' if pd.isna(x) else 'Registered')

print(f'\n✅ Cleaning complete! Final dataset: {len(df):,} rows')

## 📈 SECTION 5: Revenue Analysis

In [ ]:
# ---- Chart 1: Monthly Revenue Trend ----
monthly = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly['YearMonth'] = monthly['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['YearMonth'], monthly['Revenue'], marker='o', color='#2196F3', linewidth=2.5)
ax.fill_between(range(len(monthly)), monthly['Revenue'], alpha=0.1, color='#2196F3')
ax.set_title('Monthly Revenue Trend (2009–2011)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Revenue (£)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.xticks(range(len(monthly)), monthly['YearMonth'], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('chart1_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Look for peak months — typically November/December (holiday season)')

In [ ]:
# ---- Chart 2: Revenue by Country (Top 10) ----
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#1565C0' if i == 0 else '#42A5F5' for i in range(len(top_countries))]
bars = ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=colors[::-1])
ax.set_title('Top 10 Countries by Revenue', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Total Revenue (£)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
for bar, val in zip(bars, top_countries.values[::-1]):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'£{val:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('chart2_revenue_by_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: UK dominates — explore growth opportunities in other top countries')

## 🛍️ SECTION 6: Product Analysis

In [ ]:
# ---- Chart 3: Top 10 Products by Revenue ----
top_products = (df.groupby('Description')['Revenue']
                  .sum()
                  .sort_values(ascending=False)
                  .head(10))

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_products.index[::-1], top_products.values[::-1], color='#43A047')
ax.set_title('Top 10 Products by Revenue', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Total Revenue (£)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('chart3_top_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Focus marketing and inventory on these high-revenue products')

In [ ]:
# ---- Chart 4: Top 10 Products by Units Sold ----
top_units = (df.groupby('Description')['Quantity']
               .sum()
               .sort_values(ascending=False)
               .head(10))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_units.index[::-1], top_units.values[::-1], color='#FB8C00')
ax.set_title('Top 10 Products by Units Sold', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Total Units Sold', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('chart4_top_units.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: High units but low revenue = low-price volume drivers')

## 👥 SECTION 7: Customer Behavior Analysis

In [ ]:
# ---- Chart 5: Customer Segments (One-time vs Repeat) ----
customer_orders = (df[df['CustomerID'].notna()]
                     .groupby('CustomerID')['InvoiceNo']
                     .nunique()
                     .reset_index())
customer_orders.columns = ['CustomerID', 'TotalOrders']

def segment(n):
    if n == 1: return 'One-time Buyer'
    elif n <= 5: return 'Occasional (2-5 orders)'
    else: return 'Loyal (6+ orders)'

customer_orders['Segment'] = customer_orders['TotalOrders'].apply(segment)
segment_counts = customer_orders['Segment'].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#EF5350', '#FFA726', '#66BB6A']
wedges, texts, autotexts = ax.pie(
    segment_counts,
    labels=segment_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    textprops={'fontsize': 12}
)
ax.set_title('Customer Segments by Purchase Frequency', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('chart5_customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print(segment_counts)
print('💡 Insight: A high % of one-time buyers = opportunity to improve retention')

In [ ]:
# ---- Chart 6: Top 10 Customers by Spend ----
top_customers = (df[df['CustomerID'].notna()]
                   .groupby('CustomerID')['Revenue']
                   .sum()
                   .sort_values(ascending=False)
                   .head(10)
                   .reset_index())
top_customers['CustomerID'] = top_customers['CustomerID'].astype(int).astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(top_customers['CustomerID'], top_customers['Revenue'], color='#7B1FA2')
ax.set_title('Top 10 Customers by Total Spend', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Customer ID', fontsize=12)
ax.set_ylabel('Total Spend (£)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('chart6_top_customers.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: VIP customers — these should receive priority service and loyalty rewards')

## ⏰ SECTION 8: Time-Based Patterns

In [ ]:
# ---- Chart 7: Sales by Day of Week ----
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df.groupby('DayOfWeek')['Revenue'].sum().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(daily.index, daily.values, color='#00897B')
ax.set_title('Revenue by Day of the Week', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Total Revenue (£)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('chart7_sales_by_day.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Weekday vs weekend patterns inform when to send marketing emails')

In [ ]:
# ---- Chart 8: Sales by Hour of Day ----
hourly = df.groupby('Hour')['Revenue'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly.index, hourly.values, marker='o', color='#E53935', linewidth=2.5)
ax.fill_between(hourly.index, hourly.values, alpha=0.15, color='#E53935')
ax.set_title('Revenue by Hour of Day', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Hour (24hr)', fontsize=12)
ax.set_ylabel('Total Revenue (£)', fontsize=12)
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.savefig('chart8_sales_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Peak shopping hours = best time for flash sales and promotions')

In [ ]:
# ---- Chart 9: Revenue by Month (Seasonal Heatmap) ----
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg = df.groupby(['Year','MonthName'])['Revenue'].sum().reset_index()
pivot = monthly_avg.pivot(index='Year', columns='MonthName', values='Revenue')
pivot = pivot.reindex(columns=month_order)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Revenue Heatmap by Year and Month', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Year', fontsize=12)
plt.tight_layout()
plt.savefig('chart9_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Darker = higher revenue. Q4 is typically the strongest quarter')

## 📊 SECTION 9: Key Findings Summary

In [ ]:
# Auto-generate key stats for your findings section
total_revenue    = df['Revenue'].sum()
total_orders     = df['InvoiceNo'].nunique()
total_customers  = df['CustomerID'].nunique()
total_products   = df['Description'].nunique()
avg_order_value  = df.groupby('InvoiceNo')['Revenue'].sum().mean()
top_country      = df.groupby('Country')['Revenue'].sum().idxmax()
top_country_pct  = df.groupby('Country')['Revenue'].sum().max() / total_revenue * 100
top_month        = df.groupby('MonthName')['Revenue'].sum().idxmax()
top_product      = df.groupby('Description')['Revenue'].sum().idxmax()
peak_hour        = df.groupby('Hour')['Revenue'].sum().idxmax()

print('=' * 55)
print('       KEY FINDINGS — CAPSTONE CASE STUDY 3')
print('=' * 55)
print(f'  Total Revenue:         £{total_revenue:>12,.2f}')
print(f'  Total Orders:          {total_orders:>12,}')
print(f'  Unique Customers:      {total_customers:>12,}')
print(f'  Unique Products:       {total_products:>12,}')
print(f'  Avg Order Value:       £{avg_order_value:>12,.2f}')
print(f'  Top Country:           {top_country:>12} ({top_country_pct:.1f}% of revenue)')
print(f'  Best Sales Month:      {top_month:>12}')
print(f'  Top Product:           {top_product[:30]}')
print(f'  Peak Shopping Hour:    {peak_hour:>10}:00')
print('=' * 55)

## 💾 SECTION 10: Export Summary Files

In [ ]:
# Export cleaned dataframe
df.to_csv('online_retail_final.csv', index=False)
print('✅ Exported: online_retail_final.csv')

# Export monthly revenue summary
monthly_summary = df.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()
monthly_summary['YearMonth'] = monthly_summary['YearMonth'].astype(str)
monthly_summary.to_csv('summary_monthly.csv', index=False)
print('✅ Exported: summary_monthly.csv')

# Export top products summary
top_products_export = df.groupby('Description').agg(
    Units_Sold=('Quantity', 'sum'),
    Total_Revenue=('Revenue', 'sum')
).sort_values('Total_Revenue', ascending=False).head(20).reset_index()
top_products_export.to_csv('summary_top_products.csv', index=False)
print('✅ Exported: summary_top_products.csv')

# Export customer segments
customer_segments = customer_orders.copy()
customer_segments.to_csv('summary_customer_segments.csv', index=False)
print('✅ Exported: summary_customer_segments.csv')

print()
print('🎉 All exports complete! Download these files for your presentation.')

---
## ✅ Analysis Complete!

### Charts Created:
1. `chart1_monthly_revenue.png` — Revenue trend over time
2. `chart2_revenue_by_country.png` — Top 10 countries
3. `chart3_top_products.png` — Top 10 products by revenue
4. `chart4_top_units.png` — Top 10 products by units sold
5. `chart5_customer_segments.png` — One-time vs repeat customers
6. `chart6_top_customers.png` — Top 10 VIP customers
7. `chart7_sales_by_day.png` — Revenue by day of week
8. `chart8_sales_by_hour.png` — Revenue by hour of day
9. `chart9_heatmap.png` — Seasonal revenue heatmap

### Next Step → Step 5: SHARE
Use these charts in your Google Slides or Tableau presentation!

---
*Google Data Analytics Professional Certificate — Capstone Case Study 3*